
Topic: Next Order Comparison Using LEAD()

Purpose:
This script demonstrates how to compare each order with the next order
for the same customer using the LEAD() window function.

Helps for event sequence analysis, customer behavior tracking, gap analysis, and next-step
comparison.



In [0]:
%sql

DROP TABLE IF EXISTS orders;

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    total_amount DECIMAL(10, 2) NOT NULL
);

INSERT INTO orders (order_id, customer_id, order_date, total_amount) VALUES
(1001, 101, '2026-01-01', 250.00),
(1002, 101, '2026-01-05', 400.00),
(1003, 101, '2026-01-10', 150.00),
(1004, 102, '2026-01-02', 800.00),
(1005, 102, '2026-01-06', 300.00),
(1006, 102, '2026-01-12', 500.00),
(1007, 103, '2026-01-03', 700.00),
(1008, 103, '2026-01-09', 900.00);

WITH order_comparison AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        total_amount,
        LEAD(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date, order_id
        ) AS next_order_date,
        LEAD(total_amount) OVER (
            PARTITION BY customer_id
            ORDER BY order_date, order_id
        ) AS next_order_amount
    FROM orders
)

SELECT
    order_id,
    customer_id,
    order_date,
    total_amount,
    next_order_date,
    next_order_amount,
    next_order_amount - total_amount AS next_amount_difference
FROM order_comparison
ORDER BY
    customer_id,
    order_date;

order_id,customer_id,order_date,total_amount,next_order_date,next_order_amount,next_amount_difference
1001,101,2026-01-01,250.00,2026-01-05,400.00,150.00
1002,101,2026-01-05,400.00,2026-01-10,150.00,-250.00
1003,101,2026-01-10,150.00,null,null,null
1004,102,2026-01-02,800.00,2026-01-06,300.00,-500.00
1005,102,2026-01-06,300.00,2026-01-12,500.00,200.00
1006,102,2026-01-12,500.00,null,null,null
1007,103,2026-01-03,700.00,2026-01-09,900.00,200.00
1008,103,2026-01-09,900.00,null,null,null


In [0]:
%sql
-- Define a Common Table Expression (CTE) named 'order_comparison'
-- This builds a temporary timeline looking forward into the next transaction
WITH order_comparison AS (
    SELECT 
        order_id,
        customer_id,
        order_date,
        total_amount,
        
        -- LEAD looks forward to grab the date of the next chronological order
        -- PARTITION BY groups data by customer so we don't look into another customer's history
        -- ORDER BY ensures the timeline runs chronologically from oldest to newest
        LEAD(order_date) OVER (
            PARTITION BY customer_id 
            ORDER BY order_date, order_id
        ) AS next_order_date,
        
        -- LEAD looks forward to grab the dollar amount of the next chronological order
        LEAD(total_amount) OVER (
            PARTITION BY customer_id 
            ORDER BY order_date, order_id
        ) AS next_order_amount
    FROM orders
)
-- Main query to calculate the forward-looking financial variance
SELECT 
    order_id,
    customer_id,
    order_date,
    total_amount,
    next_order_date,
    next_order_amount,
    
    -- Calculate the dollar difference: (Next Order Amount) - (Current Order Amount)
    -- Positive numbers mean the customer spent more on their next visit.
    -- Negative numbers mean the customer spent less on their next visit.
    -- NULL means this is the customer's final/most recent order (no next order exists).
    next_order_amount - total_amount AS next_amount_difference
FROM order_comparison
-- Sort the final output cleanly by customer and their order timeline
ORDER BY 
    customer_id,
    order_date;